# 03 · Engine sandbox (Phase 3)

Scratch space for the Monte Carlo engine. Each function gets tried here first, then moves into `model/simulate.py` once it works.

Method for every function: **predict the output first, then run it.** Fixed seed so results are reproducible.

In [ ]:
import numpy as np
import pandas as pd

SEED = 42
N_RUNS = 10_000
rng = np.random.default_rng(SEED)
DATA = "../data/"   # notebooks/ sits one level below the project root

## Function 1: presets, parameter loader, sampler
**What it does:** holds the three dial presets, loads `parameters.csv`, and draws one coefficient per link, uniform in [low, high].

**My prediction before running:** 29 coefficients, every one inside its own range; 3 presets with the same 6 dial keys.

In [ ]:
PRESETS = {
    "Actual (documented)": {
        "Physical discipline intensity": 9, "Emotional warmth & affection": 3,
        "Autonomy granted": 2, "Normalcy (school peers play)": 2,
        "Body-image comments (negative)": 8, "Parental financial control": 9},
    "Strict coach, warm father": {
        "Physical discipline intensity": 5, "Emotional warmth & affection": 6,
        "Autonomy granted": 4, "Normalcy (school peers play)": 4,
        "Body-image comments (negative)": 4, "Parental financial control": 6},
    "Firm but nurturing (counterfactual)": {
        "Physical discipline intensity": 2, "Emotional warmth & affection": 8,
        "Autonomy granted": 7, "Normalcy (school peers play)": 7,
        "Body-image comments (negative)": 1, "Parental financial control": 4},
}


def load_parameters(path="data/parameters.csv"):
    return pd.read_csv(path)


def sample_params(params, rng):
    """One draw: a coefficient per link, uniform in [low, high]."""
    return dict(zip(params.link_id, rng.uniform(params.low, params.high)))


params = load_parameters(DATA + "parameters.csv")
c = sample_params(params, rng)
inside = all(r.low <= c[r.link_id] <= r.high for r in params.itertuples())
print(len(c), "coefficients; all inside their ranges:", inside)
print({k: round(v, 3) for k, v in list(c.items())[:3]})
print({name: sorted(d) == sorted(PRESETS["Actual (documented)"]) for name, d in PRESETS.items()})

## Function 2: mediators from dials
**What it does:** turns the six dials into six mediators (0-10). Practice is a level; the other five are loads.

**My prediction before running** (Actual preset, range midpoints, worked by hand):
trauma = 10 x (0.60 x 0.9 - 0.40 x 0.3 + 0.25 x 0.8) = **6.2**; practice = 10 x (0.25 + 0.50 x 0.9 + 0.25 x 0.3 + 0.175 x 0.2) = **8.1**; self-image **3.575**; attachment **3.675**; sleep **2.425**; financial delay **4.225**.

In [ ]:
def compute_mediators(dials, c):
    """Dials (0-10) -> six mediators (0-10, clipped). Practice is a LEVEL;
    the other five are LOADS (higher = heavier). Intercepts 0.25 (practice)
    and 0.55 (attachment) are ASSUMPTION baselines."""
    d = {k: v / 10 for k, v in dials.items()}
    trauma = 10 * (c["P01"] * d["Physical discipline intensity"]
                   + c["P02"] * d["Emotional warmth & affection"]
                   + c["P03"] * d["Body-image comments (negative)"])
    practice = 10 * (0.25
                     + c["P04"] * d["Physical discipline intensity"]
                     + c["P05"] * d["Emotional warmth & affection"]
                     + c["P06"] * d["Autonomy granted"])
    self_image = 10 * (c["P07"] * d["Body-image comments (negative)"]
                       + c["P08"] * d["Emotional warmth & affection"])
    attachment = 10 * (0.55
                       + c["P09"] * d["Emotional warmth & affection"]
                       + c["P10"] * d["Autonomy granted"])
    sleep = 10 * (c["P11"] * d["Physical discipline intensity"]
                  + c["P12"] * d["Normalcy (school peers play)"])
    fin_delay = 10 * (c["P13"] * d["Parental financial control"]
                      + c["P14"] * d["Autonomy granted"])
    clip = lambda x: float(np.clip(x, 0, 10))
    return {"Practice hours & skill acquisition": clip(practice),
            "Trauma / fear load": clip(trauma),
            "Self-image & body perception": clip(self_image),
            "Attachment & trust in adults": clip(attachment),
            "Sleep & physical health": clip(sleep),
            "Financial independence timing": clip(fin_delay)}


# Range midpoints: one deterministic "typical draw", so the output can be predicted by hand
params = load_parameters(DATA + "parameters.csv")
mid = dict(zip(params.link_id, (params.low + params.high) / 2))
med = compute_mediators(PRESETS["Actual (documented)"], mid)
{k: round(v, 3) for k, v in med.items()}